# Initial Sync Test

# Notes

- Uses the Python Client: Furhat Realtime API, v0.1.3, 27/10/25: https://pypi.org/project/furhat-realtime-api/
- NOT the Python Client: Furhat Remote API, v1.0.2, 15/11/21: https://pypi.org/project/furhat-remote-api/#description

## Virtual Furhat Setup

- SDK for: i) Launcher to run the Virtual Furhat; ii) Virtual Furhat is the simulation.
- Follow instructions to create SDK account, dowlnload launcher and get API key: https://docs.furhat.io/setup/sdk
- Run the Launcher for the Furhat Studio: i) Virtual Furhat should run; ii) Web interface for custom control

- Websocket is: ws://<ROBOT_IP>:9000/v1/events
- Websocket playground using: http://127.0.0.1:9000/v1/index.html

## Furhat Realtime API

- Realtime API Intro: https://docs.furhat.io/realtime-api/intro
- Python client package (and API reference): https://pypi.org/project/furhat-realtime-api/
- API documentation: i) PyPi package; ii) Websocket playground; iii) API examples, https://github.com/FurhatRobotics/realtime-api-examples/tree/main/python

## Artifical Social Agent (ASA) Application

- A UV managed and pacjaged Python application
- TBC: setup instructions


In [ ]:
# Launch with all debugs

import argparse
import logging

# from asa import Greeter, __version__
from asa._tools import depreport, portcheck
from asa._tools.custom_logging import setup_logging

# from asa._tools.depreport import report
# from asa._tools.portcheck import report

log = logging.getLogger(f"{__name__}.app")

# Dummy CLI args (mirrors asa.cli's argparse.Namespace)
args = argparse.Namespace(log="debug")

# Establish custom logging
setup_logging(level=args.log.upper())

# Launch and versions
# print(Greeter("Artificial Social Agent").greet())
# print(f"ASA Version: {__version__}")
depreport.report()

# Quick test of logging
# log.info("ASA Launch")
log.debug("Log test Debug")
log.info("Log test Info")
log.warning("Log test Warning")

# Double-check kernels and ports
print("Ports Check")
portcheck.report()

In [ ]:
# from asa._tools import portcheck

# portcheck.report()               # what's alive, what's litter, who holds port 9000
# portcheck.clean()                # dry run — counts what would go, deletes nothing
# portcheck.clean(dry_run=False)   # actually delete

---

In [ ]:
# Get the Furhat package - Basic synchronous

import logging

from furhat_realtime_api import FurhatClient

test_furhat = FurhatClient(host="127.0.0.1")
test_furhat.set_logging_level(logging.DEBUG)

test_furhat.connect()
test_furhat.request_speak_text("Hello, this is a second test")
test_furhat.disconnect()

In [ ]:
# Get the Furhat package -  Asynchronous

# Websocket connection to: ws://127.0.0.1:9000/v1/events

# import asyncio NB: not needed in the notebook
import logging

from furhat_realtime_api import AsyncFurhatClient

test_furhat = AsyncFurhatClient(host="127.0.0.1")
test_furhat.set_logging_level(logging.DEBUG)

async def run_test():
    await test_furhat.connect()
    await test_furhat.request_speak_text("Hello, this is a second test",
                                         wait=True, abort=True)
    await test_furhat.disconnect()

# asyncio.run(run_test()) NB: not needed in the notebook
await run_test()



---

In [ ]:
# Get the Furhat package -  Asynchronous

# Websocket connection to: ws://127.0.0.1:9000/v1/events

import asyncio  # NB: not needed in the notebook
import logging

from furhat_realtime_api import AsyncFurhatClient, Events

test_furhat = AsyncFurhatClient(host="127.0.0.1")
test_furhat.set_logging_level(logging.DEBUG)


In [ ]:
# Start the connection / conversation session

async def on_speak_start(event):
    print("Furhat started speaking:", event)

async def on_speak_end(event):
    print("Furhat finished speaking:", event)

# await test_furhat.connect()

try:
    await test_furhat.connect()
except Exception as e:
    print(f"Connection failed with: {e}")

test_furhat.add_handler(Events.response_speak_start, on_speak_start)
test_furhat.add_handler(Events.response_speak_end, on_speak_end)
# test_furhat.add_handler([Events.response_speak_start, Events.response_speak_end], on_speak)

In [ ]:
# Multiple actions ...



async def run_actions(text):
    await test_furhat.request_speak_text(text=text, wait=True, abort=True)

# asyncio.run(run_test()) NB: not needed in the notebook
await run_actions("Hello, this is a third test")

# await test_furhat.request_led_set(color="blue")
await test_furhat.request_gesture_start(name="Smile", intensity=0.6, duration=2.0)

In [ ]:
await asyncio.gather(
    test_furhat.request_speak_text("Lovely to see you", wait=True, abort=True),
    test_furhat.request_gesture_start(name="BigSmile", intensity=0.9, duration=4.0, wait=True),
    test_furhat.request_gesture_start(name="Blink", intensity=0.8, duration=6.0, wait=True),
)

In [ ]:
# End the connection / conversation session

test_furhat.event_handlers.clear() 
await test_furhat.disconnect()

---

In [ ]:
from asa import ASASession

session = await ASASession().start()      # keep it open across cells


In [ ]:
await session.say("Hello, this is a second test")

In [ ]:
await session.stop()

---

## Future use

This is why the async client is the right one for your prototype:

- asyncio.gather(...) — speak and gesture and set the LED as one turn, rather than three round-trips end to end.
- asyncio.wait_for(coro, timeout) — the client already uses it internally with a 5s default. A user who says nothing shouldn't hang your agent.
- task.cancel() — barge-in. User starts talking while Furhat is mid-sentence, you cancel the speaking task. The abort=True flag you're passing is the protocol-level version of the same idea.

Handling perception while acting — the listener task means you can react to a response_hear event that arrives during an utterance. Impossible if your thread is blocked inside a speak call.

In [ ]:
import asyncio
import time


async def job(name, seconds):
    print(f"{name} start   {time.strftime('%H:%M:%S')}")
    await asyncio.sleep(seconds)
    print(f"{name} done    {time.strftime('%H:%M:%S')}")
    return name

results = await asyncio.gather(job("A", 3), job("B", 1), job("C", 2))
print(results, "— elapsed ~3s, not 6")